# ATC Multi-Agent GRPO Training on Google Colab

This notebook follows the current repo workflow:

- clone the repository into Colab
- install the training dependencies used by `training/train_grpo.py`
- build a tiny dataset sanity check
- train a LoRA adapter with GRPO
- evaluate the checkpoint against the heuristic baseline
- save reward curves and plots

Recommended runtime:

- `Runtime -> Change runtime type -> GPU`
- T4 works for small runs
- L4 or A100 is better for longer training


## 1. Configure Colab paths and training settings

Set `REPO_URL` to your GitHub fork or repository URL before running the clone cell.


In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
REPO_DIR = Path("/content/ATC")
OUTPUT_DIR = Path("/content/drive/MyDrive/atc-multiagent") if USE_DRIVE else Path("/content/atc-multiagent")

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
EPISODES = 50
LORA_RANK = 16
SEED = 42
EVAL_EPISODES = 5

os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"REPO_URL={REPO_URL}")
print(f"REPO_DIR={REPO_DIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"EPISODES={EPISODES}")


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {OUTPUT_DIR}")


## 2. Clone the repo

If `REPO_URL` still contains the placeholder, edit the config cell above first.


In [ ]:
import shutil
import subprocess
import sys

if "<your-user>" in REPO_URL or "<your-repo>" in REPO_URL:
    raise ValueError("Set REPO_URL in the config cell before cloning.")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")


## 3. Install dependencies

This notebook pins `trl==0.9.6` to match the current training script.


In [ ]:
import subprocess
import sys

def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

pip_install("-U", "pip")
pip_install(
    "unsloth[colab-new]",
    "trl==0.9.6",
    "datasets>=2.20.0",
    "accelerate>=0.32.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.9.0",
    "numpy>=1.26.0",
    "wandb",
    "openenv-core[core]>=0.2.3",
    "fastapi>=0.128.0",
    "openai>=2.30.0",
    "pydantic>=2.12.0",
    "uvicorn>=0.41.0",
)


## 4. Sanity-check the runtime and dataset builder


In [ ]:
import torch
from training.dataset import build_episode_dataset

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

sample_data = build_episode_dataset(n_episodes=2, seed=SEED)
print("samples:", len(sample_data))
print("roles:", sorted({row["agent_role"] for row in sample_data}))
print("sample keys:", sorted(sample_data[0].keys()))


## 5. Train the model

For a first pass, keep `EPISODES` small such as `25` or `50`.


In [ ]:
import subprocess
import sys

train_cmd = [
    sys.executable,
    "training/train_grpo.py",
    "--model", MODEL_NAME,
    "--episodes", str(EPISODES),
    "--lora_rank", str(LORA_RANK),
    "--seed", str(SEED),
    "--output_dir", str(OUTPUT_DIR),
]

print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)


## 6. Evaluate the trained checkpoint


In [ ]:
import subprocess
import sys

eval_json = OUTPUT_DIR / "eval_results.json"
eval_cmd = [
    sys.executable,
    "training/eval.py",
    "--base", "heuristic-baseline",
    "--trained", str(OUTPUT_DIR),
    "--episodes", str(EVAL_EPISODES),
    "--output", str(eval_json),
]

print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)
print(f"Saved eval results to {eval_json}")


## 7. Plot reward curves and evaluation charts


In [ ]:
import subprocess
import sys
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

curve_cmd = [
    sys.executable,
    "training/plot_rewards.py",
    "--input", str(OUTPUT_DIR / "reward_curves.json"),
    "--save", str(plots_dir),
    "--no_show",
]
subprocess.run(curve_cmd, check=True)

eval_plot_cmd = [
    sys.executable,
    "training/plot_rewards.py",
    "--eval_results", str(OUTPUT_DIR / "eval_results.json"),
    "--save", str(plots_dir),
    "--no_show",
]
subprocess.run(eval_plot_cmd, check=True)

for name in ["training_curves.png", "eval_comparison.png"]:
    path = plots_dir / name
    if path.exists():
        display(Image(filename=str(path)))


## Troubleshooting

- If Colab disconnects, keep `OUTPUT_DIR` on Drive.
- If training crashes with an out-of-memory error, reduce `EPISODES` for shorter runs or switch to a larger GPU.
- If you see a `GRPOConfig` keyword error, make sure the install cell completed with `trl==0.9.6`.
- If you want to upload the adapter later, use the saved folder in `OUTPUT_DIR` as your checkpoint.
